# Patient Propensity Scoring Tutorial

This notebook walks through building a patient collectibility model — predicting the probability that a patient will pay their out-of-pocket balance.

This powers the `get_patient_propensity` MCP tool. The production model is more sophisticated, but this reference illustrates the core GBM approach and why tree models outperform logistic regression on claims data.

**What you'll learn:**
- Why gradient boosting handles claims data better than linear models
- Feature engineering from publicly-derivable patient and claim attributes
- How to calibrate a propensity model so scores mean what they say
- Using SHAP to explain individual predictions to billing staff

**What this does NOT include:**
- PHI or real patient data
- Credit bureau features (available in production via Experian integration)
- The full 40+ feature production model

**Prerequisites:**
- `pip install -r requirements.txt` from the repo root
- No external data needed — uses synthetic claims generator

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, brier_score_loss

import sys
sys.path.insert(0, '..')

from reference.aba_auth_predictor import build_propensity_features, train_propensity_model, explain_prediction
from sample_claims_schema import generate_synthetic_claims

print('Dependencies loaded.')

## 1. Generate Synthetic Patient Data

We generate claims with attached patient attributes. In real 835 data, some of these come from the claim itself; others come from patient intake forms or insurance eligibility responses.

Features used in this reference:
- `balance_amount`: out-of-pocket balance owed
- `days_since_service`: how long the balance has been outstanding
- `insurance_type`: commercial vs Medicaid vs Medicare
- `copay_collected_at_service`: binary — was copay collected at time of service?
- `prior_payment_history`: patient's historical payment rate on prior balances (0–1)
- `specialty`: service specialty (ABA, PT/OT, SNF, etc.)
- `payer_cluster`: the payer's behavioral cluster from notebook 01

In [ ]:
df_claims = generate_synthetic_claims(n_rows=3000, n_payers=6, include_patient_attrs=True, seed=99)

print(f'Total claims: {len(df_claims):,}')
print(f'Payment rate: {df_claims.patient_paid.mean():.1%}')
df_claims[['payer_name', 'balance_amount', 'days_since_service', 'insurance_type', 
            'prior_payment_history', 'patient_paid']].head(8)

## 2. Feature Engineering

Raw claim fields need transformation. The key non-linear relationships:

- **Balance × days**: a $50 balance at 180 days is harder to collect than a $500 balance at 7 days
- **Insurance type**: Medicaid patients have different collection dynamics than commercial
- **Prior history**: exponential decay — distant prior history matters less

Tree models handle these automatically, but good feature engineering still helps calibration.

In [ ]:
X, y, feature_names = build_propensity_features(df_claims)

print(f'Feature matrix: {X.shape}')
print(f'Target: {y.mean():.1%} positive (paid)')
print(f'Features: {feature_names}')

## 3. Train the GBM

We use gradient boosting (LightGBM under the hood) with temporal validation — we train on earlier claims and test on later ones. This mirrors the real deployment where the model is always predicting on future patients it hasn't seen.

**Why not random split?** With healthcare data, a random split leaks information: the model sees patient A's June claim in training and July claim in test. Temporal split prevents this.

In [ ]:
model, metrics = train_propensity_model(X, y, df_claims['service_date'])

print('Training complete.')
print(f"AUC (temporal CV): {metrics['auc_mean']:.4f} ± {metrics['auc_std']:.4f}")
print(f"Brier score: {metrics['brier_score']:.4f}")
print(f"  (lower is better; 0.0 = perfect, 0.25 = uninformative)")

## 4. Calibration Check

AUC tells you the model *ranks* patients correctly. Calibration tells you whether the *scores mean what they say* — a score of 0.7 should mean roughly 70% of patients at that score actually pay.

Poorly calibrated models can mislead billing staff into chasing patients who look collectible but aren't. Production Upstream uses Platt scaling to calibrate.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False  # temporal split
)

y_prob = model.predict_proba(X_test)[:, 1]
fraction_of_positives, mean_predicted_value = calibration_curve(y_test, y_prob, n_bins=10)

fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor('#0A1628')
ax.set_facecolor('#0D1B2A')

ax.plot([0, 1], [0, 1], '--', color='#FFFFFF44', label='Perfect calibration')
ax.plot(mean_predicted_value, fraction_of_positives, 's-', color='#F5A623', label='Model')

ax.set_xlabel('Mean Predicted Probability', color='#FFFBF5')
ax.set_ylabel('Fraction Who Actually Paid', color='#FFFBF5')
ax.set_title('Calibration Curve — Does the score mean what it says?', color='#FFFBF5', pad=12)
ax.tick_params(colors='#FFFBF5')
for spine in ax.spines.values():
    spine.set_edgecolor('#FFFFFF33')
ax.legend(facecolor='#0D1B2A', edgecolor='#FFFFFF33', labelcolor='#FFFBF5')

plt.tight_layout()
plt.show()

print(f'Brier score: {brier_score_loss(y_test, y_prob):.4f}')

## 5. SHAP Explainability

Billing staff don't trust a black box. SHAP values let you say *why* a patient scored high or low — which specific factors pushed the score up or down.

This is how the production `get_patient_propensity` tool returns `risk_factors` alongside the score.

In [ ]:
# Explain a single patient prediction
sample_patient = X_test[0:1]
score = model.predict_proba(sample_patient)[0, 1]

explanation = explain_prediction(model, sample_patient, feature_names)

print(f'Collectibility score: {score:.1%}')
print()
print('Top factors pushing score UP (increases collectibility):')
for feat, val in explanation.positive_factors[:3]:
    print(f'  + {feat}: {val:+.3f}')

print()
print('Top factors pushing score DOWN (reduces collectibility):')
for feat, val in explanation.negative_factors[:3]:
    print(f'  - {feat}: {val:+.3f}')

## 6. Score Distribution by Recommended Action

Raw scores are most useful when translated into concrete actions. Production Upstream segments patients into collection strategy buckets.

In [ ]:
scores = model.predict_proba(X_test)[:, 1]

def score_to_action(score: float) -> str:
    if score >= 0.75:
        return 'High — standard statement cycle'
    elif score >= 0.45:
        return 'Medium — early follow-up call at 30 days'
    elif score >= 0.20:
        return 'Low — offer payment plan proactively'
    else:
        return 'Very low — escalate to financial counselor'

score_df = pd.DataFrame({'score': scores, 'actual': y_test})
score_df['action'] = score_df['score'].apply(score_to_action)

summary = score_df.groupby('action').agg(
    n=('score', 'count'),
    avg_score=('score', 'mean'),
    actual_payment_rate=('actual', 'mean'),
).round(3)

print('Recommended actions and actual payment rates:')
print(summary.to_string())

## 7. What Production Does Differently

1. **Credit bureau integration**: Experian TrueRisk adds income estimation and credit utilization. Adds ~4–6 AUC points.

2. **Payer cluster as feature**: The patient's payer cluster from notebook 01 is a strong propensity signal — patients on Aggressive Denier payers often have higher balance anxiety and lower collectibility.

3. **Practice-specific calibration**: A $200 balance at a pediatric PT clinic collects differently than the same balance at a SNF. Production fits specialty-specific calibration layers.

4. **Real-time scoring**: The production API scores in <50ms at claim submission time, before the patient leaves.

This reference scores ~500 records/second on a single CPU core, which is sufficient for batch processing but not real-time.